# 👤 HỆ THỐNG NHẬN DIỆN KHUÔN MẶT BẰNG AI (CNN)
Chào mừng bạn đến với mô hình Nhận Diện Khuôn Mặt! Mô hình này sử dụng mạng Nơ-ron Tích chập (CNN) được xây dựng từ đầu để phân loại các khuôn mặt khác nhau.

### 📌 Hướng dẫn chuẩn bị dữ liệu:
1. Bạn đang có thư mục được chia sẻ tên là **'Ảnh selfie cho chiều t2...'** trên Google Drive.
2. Hãy click chuột phải vào thư mục đó -> Chọn **Thêm lối tắt vào Drive (Add shortcut to Drive)** -> Đổi tên lối tắt đó thành **`SELFIE`** và để ở thư mục gốc của Drive của bạn.


In [ ]:
# 1. KẾT NỐI VỚI GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.5. COPY DỮ LIỆU TỪ DRIVE VÀO Ổ CỨNG ẢO CỦA COLAB
import shutil
import os
import subprocess

drive_path = '/content/drive/MyDrive/SELFIE'
local_path = '/content/SELFIE_LOCAL'

# Bước quan trọng: Tìm ra đường dẫn THẬT của thư mục (xuyên qua Shortcut)
real_path = os.path.realpath(drive_path)
print(f"📂 Đường dẫn Shortcut: {drive_path}")
print(f"📂 Đường dẫn THẬT: {real_path}")

# Xóa dữ liệu cũ bị lỗi (nếu có)
if os.path.exists(local_path):
    shutil.rmtree(local_path)
    print("🗑️ Đã xóa sạch dữ liệu cũ.")

print("\n⏳ Đang copy dữ liệu bằng lệnh hệ thống (cp -r)...")
print("(Cách này xuyên phá được rào cản của Google Drive)\n")

# Dùng lệnh cp của Linux (mạnh hơn Python shutil rất nhiều trên Google Drive)
result = subprocess.run(
    ['cp', '-r', '--no-preserve=mode,ownership', real_path, local_path],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✅ Copy thành công 100%!")
else:
    print(f"⚠️ Có một số lỗi nhỏ (đã tự bỏ qua):")
    print(result.stderr[:500] if result.stderr else "Không có lỗi")

# Thống kê kết quả
total = 0
for person in sorted(os.listdir(local_path)):
    p = os.path.join(local_path, person)
    if os.path.isdir(p):
        count = len([f for f in os.listdir(p) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        total += count
        print(f"  ✅ {person}: {count} ảnh")

print(f"\n🎉 TỔNG CỘNG: {total} ảnh / {len(os.listdir(local_path))} người")


In [ ]:
# 2. KHAI BÁO THƯ VIỆN & TIỀN XỬ LÝ DỮ LIỆU
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import os

# Đường dẫn ĐÃ ĐƯỢC CHUYỂN SANG Ổ CỨNG ẢO của Colab
dataset_dir = '/content/SELFIE_LOCAL'

# Kích thước ảnh chuẩn bị đưa vào AI
IMG_SIZE = (200, 200)
BATCH_SIZE = 32

print('Đang đọc dữ liệu...')
# Tải tập huấn luyện (80%)
train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset='training',
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Tải tập kiểm tra (20%)
val_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset='validation',
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_dataset.class_names
NUM_CLASSES = len(class_names)
print(f'✅ Đã tìm thấy {NUM_CLASSES} người (classes) trong thư mục!')

# Bật Cache để tăng tốc độ tải ảnh lên gấp 10 lần!
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)


In [ ]:
# 3. XÂY DỰNG MÔ HÌNH NHẬN DIỆN MẠNH NHẤT (MOBILENET-V2)
# Khởi tạo mô hình AI có sẵn đã được huấn luyện trên hàng triệu bức ảnh
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(200, 200, 3),
    include_top=False,
    weights='imagenet'
)
# Đóng băng não bộ gốc để giữ nguyên kiến thức
base_model.trainable = False

model = models.Sequential([
    # Bước 0: Nhồi nhét dữ liệu (Data Augmentation)
    layers.RandomFlip("horizontal", input_shape=(200, 200, 3)),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
    
    # Bước 1: Tiền xử lý cho MobileNet (chuyển màu về dải [-1, 1])
    layers.Rescaling(1./127.5, offset=-1),
    
    # Bước 2: Dùng siêu AI MobileNetV2 để quét đặc điểm khuôn mặt
    base_model,
    
    # Bước 3: Nén thông tin
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3), # Chống học vẹt
    
    # Bước 4: Đưa ra quyết định (Phân loại ra 35 người)
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


In [ ]:
# 4. HUẤN LUYỆN AI (TRAINING)
print('🚀 Bắt đầu huấn luyện mô hình CNN...')
# Có thể chỉnh epochs lên 50 hoặc 100 tùy thời gian bạn có
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=25
)

# Lưu lại não bộ của AI sau khi học xong
model.save('Selfie_CNN_Model.keras')
print('✅ Đã lưu mô hình thành công!')


In [ ]:
# 5. VẼ BIỂU ĐỒ ĐÁNH GIÁ
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Độ chính xác Tập Huấn luyện')
plt.plot(val_acc, label='Độ chính xác Tập Kiểm tra')
plt.legend(loc='lower right')
plt.title('Độ chính xác (Accuracy)')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Sai số Tập Huấn luyện')
plt.plot(val_loss, label='Sai số Tập Kiểm tra')
plt.legend(loc='upper right')
plt.title('Sai số (Loss)')
plt.show()


In [ ]:
# 6. GIAO DIỆN UPLOAD ẢNH ĐỂ NHẬN DIỆN
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
from PIL import Image

upload_btn = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Tải ảnh lên',
    button_style='info'
)

out = widgets.Output()

def on_upload_change(change):
    with out:
        clear_output()
        if not upload_btn.value:
            return
            
        uploaded_file = upload_btn.value[0] if isinstance(upload_btn.value, tuple) else list(upload_btn.value.values())[0]
        content = uploaded_file['content']
        
        # Hiển thị ảnh
        img = Image.open(io.BytesIO(content)).convert('RGB')
        display(img.resize((250, 250)))
        
        # Xử lý ảnh cho AI
        img_resized = img.resize(IMG_SIZE)
        img_array = tf.keras.preprocessing.image.img_to_array(img_resized)
        img_array = tf.expand_dims(img_array, 0) # Tạo batch
        
        # Dự đoán
        predictions = model.predict(img_array)
        score = predictions[0] # Không dùng tf.nn.softmax nữa vì mô hình đã có activation='softmax' ở cuối
        
        # Đảm bảo index nằm trong giới hạn
        pred_idx = np.argmax(score)
        if pred_idx < len(class_names):
            predicted_class = class_names[pred_idx]
            confidence = 100 * np.max(score)
            
            print(f'\n🤖 KẾT QUẢ DỰ ĐOÁN:')
            print(f'➤ Người trong ảnh là: {predicted_class}')
            print(f'➤ Độ tự tin (Chính xác): {confidence:.2f}%')
        else:
            print('Lỗi: Lớp dự đoán vượt quá số lượng classes!')

upload_btn.observe(on_upload_change, names='value')
print("👇 BẤM NÚT BÊN DƯỚI ĐỂ TẢI ẢNH LÊN VÀ NHẬN DIỆN 👇")
display(upload_btn, out)
